In [ ]:
!pip install transformers datasets scikit-learn pandas numpy torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1


In [ ]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
!wget https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
!unzip liar_dataset.zip

--2026-03-15 18:02:59--  https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
Resolving www.cs.ucsb.edu (www.cs.ucsb.edu)... 23.185.0.253, 2620:12a:8000::253, 2620:12a:8001::253
Connecting to www.cs.ucsb.edu (www.cs.ucsb.edu)|23.185.0.253|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://sites.cs.ucsb.edu/~william/data/liar_dataset.zip [following]
--2026-03-15 18:02:59--  https://sites.cs.ucsb.edu/~william/data/liar_dataset.zip
Resolving sites.cs.ucsb.edu (sites.cs.ucsb.edu)... 128.111.27.164
Connecting to sites.cs.ucsb.edu (sites.cs.ucsb.edu)|128.111.27.164|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1013571 (990K) [application/zip]
Saving to: ‘liar_dataset.zip.1’

liar_dataset.zip.1  100%[===================>] 989.82K  --.-KB/s    in 0.05s   

2026-03-15 18:02:59 (18.3 MB/s) - ‘liar_dataset.zip.1’ saved [1013571/1013571]

Archive:  liar_dataset.zip
replace README? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
import os
os.listdir()

['.config',
 'liar_dataset.zip.1',
 'test.tsv',
 'README',
 'liar_dataset.zip',
 'valid.tsv',
 'train.tsv',
 'sample_data']

In [ ]:
columns = [
'id','label','statement','subject','speaker',
'speaker_job','state_info','party_affiliation',
'barely_true_counts','false_counts','half_true_counts',
'mostly_true_counts','pants_on_fire_counts','context'
]

train_df = pd.read_csv("train.tsv", sep="\t", names=columns)
valid_df = pd.read_csv("valid.tsv", sep="\t", names=columns)
test_df = pd.read_csv("test.tsv", sep="\t", names=columns)

In [ ]:
label_map = {
'pants-fire':0,
'false':1,
'barely-true':2,
'half-true':3,
'mostly-true':4,
'true':5
}

train_df['label'] = train_df['label'].map(label_map)
valid_df['label'] = valid_df['label'].map(label_map)
test_df['label'] = test_df['label'].map(label_map)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
class LiarDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_enc = tokenize(train_df["statement"])
valid_enc = tokenize(valid_df["statement"])
test_enc = tokenize(test_df["statement"])

train_dataset = LiarDataset(train_enc, train_df["label"].values)
valid_dataset = LiarDataset(valid_enc, valid_df["label"].values)
test_dataset = LiarDataset(test_enc, test_df["label"].values)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='weighted'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
import transformers
print(transformers.__version__)

5.3.0


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=100,
    disable_tqdm=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset
)

In [ ]:
trainer.train()

{'loss': '1.489', 'grad_norm': '8.521', 'learning_rate': '1.897e-05', 'epoch': '0.1562'}
{'loss': '1.389', 'grad_norm': '10.69', 'learning_rate': '1.793e-05', 'epoch': '0.3125'}
{'loss': '1.318', 'grad_norm': '11.9', 'learning_rate': '1.689e-05', 'epoch': '0.4688'}
{'loss': '1.286', 'grad_norm': '13.14', 'learning_rate': '1.584e-05', 'epoch': '0.625'}
{'loss': '1.269', 'grad_norm': '10.71', 'learning_rate': '1.48e-05', 'epoch': '0.7812'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.235', 'grad_norm': '13.19', 'learning_rate': '1.376e-05', 'epoch': '0.9375'}
{'loss': '1.104', 'grad_norm': '11.45', 'learning_rate': '1.272e-05', 'epoch': '1.094'}
{'loss': '0.9656', 'grad_norm': '16.82', 'learning_rate': '1.168e-05', 'epoch': '1.25'}
{'loss': '0.8892', 'grad_norm': '15.15', 'learning_rate': '1.064e-05', 'epoch': '1.406'}
{'loss': '0.9258', 'grad_norm': '21.15', 'learning_rate': '9.594e-06', 'epoch': '1.562'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.8767', 'grad_norm': '16.95', 'learning_rate': '8.552e-06', 'epoch': '1.719'}
{'loss': '0.9384', 'grad_norm': '17.68', 'learning_rate': '7.51e-06', 'epoch': '1.875'}
{'loss': '0.8877', 'grad_norm': '14.23', 'learning_rate': '6.469e-06', 'epoch': '2.031'}
{'loss': '0.6387', 'grad_norm': '20.9', 'learning_rate': '5.427e-06', 'epoch': '2.188'}
{'loss': '0.6822', 'grad_norm': '28.03', 'learning_rate': '4.385e-06', 'epoch': '2.344'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.7044', 'grad_norm': '22.52', 'learning_rate': '3.344e-06', 'epoch': '2.5'}
{'loss': '0.7655', 'grad_norm': '30.48', 'learning_rate': '2.302e-06', 'epoch': '2.656'}
{'loss': '0.7816', 'grad_norm': '27.3', 'learning_rate': '1.26e-06', 'epoch': '2.812'}
{'loss': '0.8039', 'grad_norm': '20.65', 'learning_rate': '2.187e-07', 'epoch': '2.969'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1329', 'train_samples_per_second': '23.11', 'train_steps_per_second': '1.444', 'train_loss': '0.9957', 'epoch': '3'}


TrainOutput(global_step=1920, training_loss=0.9957131206989288, metrics={'train_runtime': 1329.4022, 'train_samples_per_second': 23.108, 'train_steps_per_second': 1.444, 'train_loss': 0.9957131206989288, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

{'eval_loss': '2.336', 'eval_runtime': '5.334', 'eval_samples_per_second': '240.7', 'eval_steps_per_second': '15.19', 'epoch': '3'}


{'eval_loss': 2.3362858295440674,
 'eval_runtime': 5.3344,
 'eval_samples_per_second': 240.704,
 'eval_steps_per_second': 15.185,
 'epoch': 3.0}

In [ ]:
trainer.evaluate(test_dataset)

{'eval_loss': '2.255', 'eval_runtime': '16.75', 'eval_samples_per_second': '75.63', 'eval_steps_per_second': '4.775', 'epoch': '3'}


{'eval_loss': 2.2546935081481934,
 'eval_runtime': 16.7529,
 'eval_samples_per_second': 75.629,
 'eval_steps_per_second': 4.775,
 'epoch': 3.0}

In [ ]:
model.save_pretrained("fake_news_model")
tokenizer.save_pretrained("fake_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fake_news_model/tokenizer_config.json', 'fake_news_model/tokenizer.json')

In [ ]:
!zip -r fake_news_model.zip fake_news_model

  adding: fake_news_model/ (stored 0%)
  adding: fake_news_model/config.json (deflated 56%)
  adding: fake_news_model/tokenizer.json (deflated 71%)
  adding: fake_news_model/tokenizer_config.json (deflated 42%)
  adding: fake_news_model/model.safetensors (deflated 7%)


In [ ]:
from google.colab import files
files.download("fake_news_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch

labels = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true"
]

def predict_news(text):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs).item()

    return labels[pred]

In [ ]:
predict_news("The economy created 5 million jobs this year")

'mostly-true'